# NB01 — Data Collection

## Project question

**How did consumer-price inflation and youth unemployment differ across Italy, Türkiye, and the United Kingdom from 2010 to the latest common year available?**

## Purpose of this notebook

This notebook collects the main project data programmatically from the World Bank Indicators API.

It will:

1. define the countries, indicators, and requested years;
2. test the API with one small request;
3. collect both indicators for all three countries;
4. inspect and validate the returned responses;
5. save the original JSON responses under `data/raw/`.

The original API responses are saved without transforming them into an analytical table. Data preparation will take place separately in `NB02-Data-Transformation.ipynb`.



## Data source

This project uses the World Bank Indicators API version 2 because it provides
comparable annual data for all three countries through a public API.
The API returns structured JSON data that can be collected directly
with Python and does not require an API key.

The selected indicators are:

- `FP.CPI.TOTL.ZG` — Inflation, consumer prices (annual %)
- `SL.UEM.1524.ZS` — Unemployment, youth total (% of the labour force
  aged 15–24), modelled ILO estimate

The 15–24 age range is part of the definition of the selected World Bank
indicator rather than a range created independently for this project.
Youth unemployment refers to unemployed people as a percentage of the
labour force aged 15–24, not the entire population in that age group.

The selected countries are Italy (`ITA`), Türkiye (`TUR`), and the United
Kingdom (`GBR`).

## Collection approach

NB01 sends one HTTP GET request for each indicator using the World Bank
country-and-indicator endpoint. Each request includes all three selected
countries, requests the period 2010–2026, asks for JSON output, and sets
`per_page=100` so that the complete project-specific response should fit
on one page.

The request is intentionally filtered to the countries, indicators, and
years required by the research question. It does not download the entire
World Bank database and remove unrelated observations afterwards.

The complete response returned for each request is saved without reshaping
under `data/raw/`. The notebook then validates the response structure,
pagination, countries, indicator code, missing values, and country-year
coverage. The latest common year is calculated separately from the
non-missing years available across all six country-indicator combinations.

In [1]:
import json
from pathlib import Path

import requests

In [2]:
API_BASE_URL = "https://api.worldbank.org/v2"

COUNTRIES = {
    "ITA": "Italy",
    "TUR": "Türkiye",
    "GBR": "United Kingdom",
}

INDICATORS = {
    "FP.CPI.TOTL.ZG": "Inflation, consumer prices (annual %)",
    "SL.UEM.1524.ZS": (
        "Unemployment, youth total "
        "(% of total labour force ages 15–24), modelled ILO estimate"
    ),
}

START_YEAR = 2010
END_YEAR = 2026

RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print("Countries:")
for code, name in COUNTRIES.items():
    print(f"- {code}: {name}")

print("\nIndicators:")
for code, name in INDICATORS.items():
    print(f"- {code}: {name}")

print(f"\nRequested years: {START_YEAR}–{END_YEAR}")
print(f"Raw data folder: {RAW_DATA_DIR.resolve()}")

Countries:
- ITA: Italy
- TUR: Türkiye
- GBR: United Kingdom

Indicators:
- FP.CPI.TOTL.ZG: Inflation, consumer prices (annual %)
- SL.UEM.1524.ZS: Unemployment, youth total (% of total labour force ages 15–24), modelled ILO estimate

Requested years: 2010–2026
Raw data folder: /files/assignments/final-project/data/raw


## Testing the endpoint and response structure

Before collecting both indicators, a smaller request for Italian inflation
is used to confirm that the endpoint succeeds and to inspect how the World
Bank organises its metadata and observation records.

In [4]:
test_country = "ITA"
test_indicator = "FP.CPI.TOTL.ZG"

test_url = (
    f"{API_BASE_URL}/country/{test_country}"
    f"/indicator/{test_indicator}"
)

test_params = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 100,
}

test_response = requests.get(
    test_url,
    params=test_params,
    timeout=(10, 60),
)

print("Status code:", test_response.status_code)
print("Requested URL:", test_response.url)

test_response.raise_for_status()

Status code: 200
Requested URL: https://api.worldbank.org/v2/country/ITA/indicator/FP.CPI.TOTL.ZG?format=json&date=2010%3A2026&per_page=100


In [5]:
test_data = test_response.json()

print("Outer object type:", type(test_data))
print("Number of outer elements:", len(test_data))

Outer object type: <class 'list'>
Number of outer elements: 2


In [6]:
test_metadata = test_data[0]
test_metadata

{'page': 1,
 'pages': 1,
 'per_page': 100,
 'total': 16,
 'sourceid': '2',
 'lastupdated': '2026-07-13'}

In [7]:
test_records = test_data[1]

print("Number of observations:", len(test_records))
test_records[:2]

Number of observations: 16


[{'indicator': {'id': 'FP.CPI.TOTL.ZG',
   'value': 'Inflation, consumer prices (annual %)'},
  'country': {'id': 'IT', 'value': 'Italy'},
  'countryiso3code': 'ITA',
  'date': '2025',
  'value': 1.5316682765282,
  'unit': '',
  'obs_status': '',
  'decimal': 1},
 {'indicator': {'id': 'FP.CPI.TOTL.ZG',
   'value': 'Inflation, consumer prices (annual %)'},
  'country': {'id': 'IT', 'value': 'Italy'},
  'countryiso3code': 'ITA',
  'date': '2024',
  'value': 0.982373023061417,
  'unit': '',
  'obs_status': '',
  'decimal': 1}]

## Full data collection

The test request confirmed the API structure and the fields returned for one country and indicator.

The full collection below requests each indicator for all three countries in a single API call. The two original response bodies are saved separately because they represent two distinct indicator requests. No analytical cleaning or reshaping is performed in this notebook.

In [8]:
COUNTRY_QUERY = ";".join(COUNTRIES)

RAW_FILENAMES = {
    "FP.CPI.TOTL.ZG": "world_bank_inflation.json",
    "SL.UEM.1524.ZS": "world_bank_youth_unemployment.json",
}

print("Country query:", COUNTRY_QUERY)

Country query: ITA;TUR;GBR


### Pagination completeness

Each indicator request can return at most 51 observations: three countries
across 17 requested years. Since `per_page` is set to 100, the complete
response should fit on one page.

The function verifies that the API reports one page and that the number of
records received matches the total reported in the metadata. This prevents
the notebook from silently continuing with an incomplete paginated response.

In [9]:
def fetch_indicator(indicator_code):
    """Request one World Bank indicator for all selected countries."""

    url = (
        f"{API_BASE_URL}/country/{COUNTRY_QUERY}"
        f"/indicator/{indicator_code}"
    )

    params = {
        "format": "json",
        "date": f"{START_YEAR}:{END_YEAR}",
        "per_page": 100,
    }

    response = requests.get(
        url,
        params=params,
        timeout=(10, 60),
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, list) or len(payload) != 2:
        raise ValueError(
            f"Unexpected API response structure for {indicator_code}."
        )

    metadata, records = payload

    if not isinstance(metadata, dict) or not isinstance(records, list):
        raise ValueError(
            f"Unexpected metadata or records for {indicator_code}."
        )

    # Confirm that the complete result was returned on one page.
    if metadata.get("page") != 1:
        raise ValueError(
            f"Expected the first response page for {indicator_code}, "
            f"but received page {metadata.get('page')}."
        )

    if metadata.get("pages") != 1:
        raise ValueError(
            f"Expected one response page for {indicator_code}, "
            f"but the API reports {metadata.get('pages')} pages."
        )

    if metadata.get("total") != len(records):
        raise ValueError(
            f"Metadata reports {metadata.get('total')} records for "
            f"{indicator_code}, but {len(records)} records were returned."
        )

    return response, payload

In [10]:
responses = {}
payloads = {}

for indicator_code, indicator_name in INDICATORS.items():
    response, payload = fetch_indicator(indicator_code)

    responses[indicator_code] = response
    payloads[indicator_code] = payload

    metadata, records = payload

    print(indicator_name)
    print("  Status code:", response.status_code)
    print("  Records returned:", len(records))
    print("  API metadata total:", metadata["total"])

Inflation, consumer prices (annual %)
  Status code: 200
  Records returned: 48
  API metadata total: 48
Unemployment, youth total (% of total labour force ages 15–24), modelled ILO estimate
  Status code: 200
  Records returned: 48
  API metadata total: 48


### Validating the returned observations

After collection, each response is checked to confirm that it contains
the requested countries and indicator. The returned year range and number
of missing values are also inspected before the latest common year is
calculated.

In [11]:
for indicator_code, payload in payloads.items():
    metadata, records = payload

    returned_countries = {
        record["countryiso3code"]
        for record in records
    }

    returned_indicators = {
        record["indicator"]["id"]
        for record in records
    }

    returned_years = sorted({
        int(record["date"])
        for record in records
    })

    missing_values = sum(
        record["value"] is None
        for record in records
    )

    print(INDICATORS[indicator_code])
    print(
        "  Countries returned:",
        sorted(returned_countries),
    )
    print(
        "  Years returned:",
        f"{min(returned_years)}–{max(returned_years)}",
    )
    print(
        "  Missing values:",
        missing_values,
    )

    if returned_countries != set(COUNTRIES):
        raise ValueError(
            f"Unexpected countries returned for {indicator_code}."
        )

    if returned_indicators != {indicator_code}:
        raise ValueError(
            f"Unexpected indicator returned for {indicator_code}."
        )

    if missing_values != 0:
        raise ValueError(
            f"{missing_values} missing values were returned "
            f"for {indicator_code}."
        )

Inflation, consumer prices (annual %)
  Countries returned: ['GBR', 'ITA', 'TUR']
  Years returned: 2010–2025
  Missing values: 0
Unemployment, youth total (% of total labour force ages 15–24), modelled ILO estimate
  Countries returned: ['GBR', 'ITA', 'TUR']
  Years returned: 2010–2025
  Missing values: 0


### Identifying the latest common year

The requested period extends through 2026, but the final analysis requires a
year with usable observations for every country and both indicators.

For each of the six country-indicator combinations, the following cell
identifies the years with non-missing values. It then takes the intersection
of those year sets. The most recent year in that intersection becomes the
latest common year used in the analysis.

In [12]:
# Collect the available years for every country-indicator combination.
coverage_year_sets = []

for indicator_code, payload in payloads.items():
    _, records = payload

    for country_code in COUNTRIES:
        available_years = {
            int(record["date"])
            for record in records
            if record["countryiso3code"] == country_code
            and record["value"] is not None
        }

        if not available_years:
            raise ValueError(
                f"No usable observations were returned for "
                f"{indicator_code}, {country_code}."
            )

        coverage_year_sets.append(available_years)

# Keep only years that appear in all six country-indicator combinations.
common_years = sorted(
    set.intersection(*coverage_year_sets)
)

if not common_years:
    raise ValueError(
        "No common year was available across all countries "
        "and indicators."
    )

LATEST_COMMON_YEAR = common_years[-1]

print("Common years available:", common_years)
print("Latest common year available:", LATEST_COMMON_YEAR)

Common years available: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Latest common year available: 2025


## Saving and verifying the raw responses

The complete response body for each indicator is written directly to
`data/raw/`. The saved JSON files are then reloaded and compared with the
responses held in memory to confirm that storage did not alter their
contents. A final check confirms the number of records per country and
the absence of duplicate country-year observations.

In [13]:
for indicator_code, response in responses.items():
    output_path = RAW_DATA_DIR / RAW_FILENAMES[indicator_code]

    output_path.write_bytes(response.content)

    print(
        f"Saved {RAW_FILENAMES[indicator_code]} "
        f"({output_path.stat().st_size:,} bytes)"
    )

Saved world_bank_inflation.json (11,065 bytes)
Saved world_bank_youth_unemployment.json (12,842 bytes)


In [14]:
for indicator_code, original_payload in payloads.items():
    input_path = RAW_DATA_DIR / RAW_FILENAMES[indicator_code]

    with input_path.open("r", encoding="utf-8-sig") as file:
        reloaded_payload = json.load(file)

    if reloaded_payload != original_payload:
        raise ValueError(
            f"Saved file does not match the API response: {input_path}"
        )

    print(f"Verified: {input_path.name}")

Verified: world_bank_inflation.json
Verified: world_bank_youth_unemployment.json


In [15]:
for indicator_code, payload in payloads.items():
    _, records = payload

    country_year_pairs = [
        (record["countryiso3code"], record["date"])
        for record in records
    ]

    records_per_country = {
        country_code: sum(
            record["countryiso3code"] == country_code
            for record in records
        )
        for country_code in COUNTRIES
    }

    duplicate_pairs = (
        len(country_year_pairs)
        - len(set(country_year_pairs))
    )

    print(INDICATORS[indicator_code])
    print("  Records per country:", records_per_country)
    print("  Duplicate country-year pairs:", duplicate_pairs)

    if duplicate_pairs != 0:
        raise ValueError(
            f"Duplicate country-year observations found for {indicator_code}."
        )

Inflation, consumer prices (annual %)
  Records per country: {'ITA': 16, 'TUR': 16, 'GBR': 16}
  Duplicate country-year pairs: 0
Unemployment, youth total (% of total labour force ages 15–24), modelled ILO estimate
  Records per country: {'ITA': 16, 'TUR': 16, 'GBR': 16}
  Duplicate country-year pairs: 0


## Collection summary

The World Bank Indicators API successfully returned inflation and youth-unemployment data for Italy, Türkiye, and the United Kingdom.

Each indicator response contained 48 observations: 16 annual records from 2010 through 2025 for each of the three countries. The validation found no missing values or duplicate country-year observations.

The original API response bodies were saved as:

- `data/raw/world_bank_inflation.json`
- `data/raw/world_bank_youth_unemployment.json`

Although the request extended through 2026, intersecting the years with non-missing observations across all six country-indicator combinations identified 2025 as the latest common year. No observations were removed, cleaned, or reshaped in NB01. Data preparation will take place separately in `NB02-Data-Transformation.ipynb`.